# Étape 3 — Sampler & Augmentations

## 1. Setup

In [13]:
import sys, os, time
from pathlib import Path

ROOT = Path.cwd()   # racine du depot (les notebooks sont a la racine)
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import transforms as T, models

from dc_utils.paths import IMAGE_DIR, ARTIFACTS_DIR
from dc_utils.training import (
    TrainConfig, train_model, evaluate, OcclusionDataset,
    default_train_tf, default_eval_tf, IMAGENET_MEAN, IMAGENET_STD,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device, '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

ARTIFACTS_DIR.mkdir(exist_ok=True, parents=True)
EXP_DIR = ROOT / 'experiments'
EXP_DIR.mkdir(exist_ok=True, parents=True)

device = cuda | NVIDIA GeForce RTX 3090


In [2]:
# Charger les splits déjà calculés à l'étape 2
df_train = pd.read_csv(ROOT / 'data' / 'train_split.csv')
df_val   = pd.read_csv(ROOT / 'data' / 'val_split.csv')
print('train:', len(df_train), '| val:', len(df_val))
print('train gender:', df_train['gender'].value_counts().to_dict())
print('val   gender:', df_val['gender'].value_counts().to_dict())

train: 89793 | val: 9977
train gender: {1.0: 60736, 0.0: 29057}
val   gender: {1.0: 6748, 0.0: 3229}


In [3]:
# Modèle : mobilenet_v3_small + tête linéaire 1 sortie + sigmoid
class OcclusionRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
        in_feat = self.backbone.classifier[-1].in_features
        self.backbone.classifier[-1] = nn.Linear(in_feat, 1)
    def forward(self, x):
        return torch.sigmoid(self.backbone(x)).squeeze(-1)

def fresh_model(seed: int = 42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    return OcclusionRegressor().to(device)

print('model factory ready')

model factory ready


In [4]:
# Stockage des résultats : un dict global, sauvegardé en CSV à la fin
RESULTS: list[dict] = []

def log_run(name: str, history: pd.DataFrame, best: dict, notes: str = ''):
    history.to_csv(EXP_DIR / f'history_{name}.csv', index=False)
    RESULTS.append({
        'name': name,
        'best_epoch': best.get('epoch'),
        'val_score': best.get('score'),
        'err_F': best.get('err_F'),
        'err_M': best.get('err_M'),
        'gap': best.get('gap'),
        'notes': notes,
    })
    print(f'\n>>> {name}: val_score = {best["score"]:.5f} (epoch {best["epoch"]})\n')

print('logger ready')

logger ready


## 3. Expérience A — early stopping (patience=3)

stoppe dès que val n'améliore plus pendant 3 epochs.

In [6]:
model = fresh_model()
cfg = TrainConfig(epochs=12, early_stop_patience=3)
hist, best = train_model(
    model, df_train, df_val, IMAGE_DIR, cfg, device,
    save_path=ARTIFACTS_DIR / 'exp_A_earlystop.pt', log_prefix='[A] ',
)
log_run('A_earlystop', hist, best, notes='patience=3, max 12 ep')

[A] ep 1 | loss 0.00288 | score 0.00235 | err_F 0.00128 | err_M 0.00199 | gap 0.00071 | 134s *


[A] ep 2 | loss 0.00153 | score 0.00254 | err_F 0.00227 | err_M 0.00245 | gap 0.00018 | 132s


[A] ep 3 | loss 0.00142 | score 0.00223 | err_F 0.00102 | err_M 0.00183 | gap 0.00081 | 132s *


[A] ep 4 | loss 0.00109 | score 0.00246 | err_F 0.00142 | err_M 0.00212 | gap 0.00070 | 139s


[A] ep 5 | loss 0.00093 | score 0.00143 | err_F 0.00089 | err_M 0.00125 | gap 0.00036 | 150s *


...

[A] ep 8 | loss 0.00063 | score 0.00157 | err_F 0.00067 | err_M 0.00127 | gap 0.00060 | 132s


[A] ep 9 | loss 0.00054 | score 0.00132 | err_F 0.00061 | err_M 0.00109 | gap 0.00048 | 133s *


[A] ep10 | loss 0.00046 | score 0.00131 | err_F 0.00058 | err_M 0.00107 | gap 0.00049 | 133s *


[A] ep11 | loss 0.00041 | score 0.00129 | err_F 0.00057 | err_M 0.00105 | gap 0.00049 | 133s *


[A] ep12 | loss 0.00038 | score 0.00132 | err_F 0.00057 | err_M 0.00107 | gap 0.00050 | 135s

>>> A_earlystop: val_score = 0.00129 (epoch 11)



## 4. Expérience B — WeightedRandomSampler

In [7]:
model = fresh_model()
cfg = TrainConfig(epochs=8, sampler_mode='metric')
hist, best = train_model(
    model, df_train, df_val, IMAGE_DIR, cfg, device,
    save_path=ARTIFACTS_DIR / 'exp_B_sampler.pt', log_prefix='[B] ',
)
log_run('B_sampler', hist, best, notes='WeightedRandomSampler, weights = 1/30 + GT')

[B] ep 1 | loss 0.00297 | score 0.00232 | err_F 0.00140 | err_M 0.00201 | gap 0.00062 | 163s *


[B] ep 2 | loss 0.00120 | score 0.00196 | err_F 0.00141 | err_M 0.00178 | gap 0.00037 | 162s *


[B] ep 3 | loss 0.00088 | score 0.00765 | err_F 0.00355 | err_M 0.00629 | gap 0.00273 | 147s


[B] ep 4 | loss 0.00106 | score 0.00194 | err_F 0.00094 | err_M 0.00160 | gap 0.00066 | 159s *


[B] ep 5 | loss 0.00060 | score 0.00136 | err_F 0.00071 | err_M 0.00114 | gap 0.00043 | 149s *


[B] ep 6 | loss 0.00046 | score 0.00135 | err_F 0.00067 | err_M 0.00112 | gap 0.00045 | 165s *


[B] ep 7 | loss 0.00039 | score 0.00126 | err_F 0.00065 | err_M 0.00106 | gap 0.00041 | 153s *


[B] ep 8 | loss 0.00035 | score 0.00128 | err_F 0.00065 | err_M 0.00107 | gap 0.00042 | 144s

>>> B_sampler: val_score = 0.00126 (epoch 7)



## 5. Expérience C — augmentations

In [8]:
train_tf_aug = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.02),
    T.RandomAffine(degrees=10, translate=(0.05, 0.05), fill=0),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

model = fresh_model()
cfg = TrainConfig(epochs=8, train_tf=train_tf_aug)
hist, best = train_model(
    model, df_train, df_val, IMAGE_DIR, cfg, device,
    save_path=ARTIFACTS_DIR / 'exp_C_aug.pt', log_prefix='[C] ',
)
log_run('C_aug', hist, best, notes='ColorJitter + RandomAffine')

[C] ep 1 | loss 0.00316 | score 0.00227 | err_F 0.00143 | err_M 0.00199 | gap 0.00056 | 319s *


[C] ep 2 | loss 0.00176 | score 0.00268 | err_F 0.00186 | err_M 0.00241 | gap 0.00055 | 307s


[C] ep 3 | loss 0.00153 | score 0.00200 | err_F 0.00114 | err_M 0.00171 | gap 0.00057 | 308s *


[C] ep 4 | loss 0.00125 | score 0.00170 | err_F 0.00089 | err_M 0.00143 | gap 0.00054 | 288s *


[C] ep 5 | loss 0.00105 | score 0.00135 | err_F 0.00076 | err_M 0.00116 | gap 0.00040 | 282s *


[C] ep 6 | loss 0.00093 | score 0.00143 | err_F 0.00077 | err_M 0.00121 | gap 0.00044 | 294s


[C] ep 7 | loss 0.00078 | score 0.00125 | err_F 0.00064 | err_M 0.00105 | gap 0.00041 | 304s *


[C] ep 8 | loss 0.00069 | score 0.00120 | err_F 0.00064 | err_M 0.00102 | gap 0.00037 | 302s *

>>> C_aug: val_score = 0.00120 (epoch 8)



## 6. Expérience D — A + B + C combinés

In [9]:
model = fresh_model()
cfg = TrainConfig(
    epochs=12,
    early_stop_patience=3,
    sampler_mode='metric',
    train_tf=train_tf_aug,
)
hist, best = train_model(
    model, df_train, df_val, IMAGE_DIR, cfg, device,
    save_path=ARTIFACTS_DIR / 'exp_D_combined.pt', log_prefix='[D] ',
)
log_run('D_combined', hist, best, notes='A+B+C')

[D] ep 1 | loss 0.00315 | score 0.00226 | err_F 0.00117 | err_M 0.00190 | gap 0.00073 | 288s *


[D] ep 2 | loss 0.00150 | score 0.00184 | err_F 0.00163 | err_M 0.00177 | gap 0.00014 | 278s *


[D] ep 3 | loss 0.00138 | score 0.00171 | err_F 0.00103 | err_M 0.00148 | gap 0.00045 | 278s *


[D] ep 4 | loss 0.00100 | score 0.00207 | err_F 0.00142 | err_M 0.00185 | gap 0.00043 | 294s


[D] ep 5 | loss 0.00096 | score 0.00129 | err_F 0.00077 | err_M 0.00112 | gap 0.00035 | 299s *


...

[D] ep 8 | loss 0.00055 | score 0.00120 | err_F 0.00071 | err_M 0.00104 | gap 0.00033 | 268s


[D] ep 9 | loss 0.00049 | score 0.00109 | err_F 0.00068 | err_M 0.00095 | gap 0.00028 | 267s *


[D] ep10 | loss 0.00044 | score 0.00113 | err_F 0.00065 | err_M 0.00097 | gap 0.00032 | 267s


[D] ep11 | loss 0.00040 | score 0.00109 | err_F 0.00063 | err_M 0.00093 | gap 0.00031 | 266s *


[D] ep12 | loss 0.00038 | score 0.00105 | err_F 0.00062 | err_M 0.00091 | gap 0.00029 | 266s *

>>> D_combined: val_score = 0.00105 (epoch 12)

